# Exploring tables of `bronze` layers

Before we write a single line of dbt SQL, we look at the actual data: what's missing, what's duplicated, what's inconsistent. Every finding below becomes one cleaning rule in `dbt_project/models/silver/silver_orders.sql`.

**Workflow:** explore here -> encode the rule in dbt -> validate the result (last section of this notebook, run *after* `dbt run`).

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

# Find the repo root dynamically
root_candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve().parent.parent,
]

for root in root_candidates:
    if (root / "config_loader.py").exists():
        sys.path.insert(0, str(root))
        break
else:
    raise FileNotFoundError("Could not find config_loader.py in the project tree.")

from config_loader import CONFIG

engine = create_engine(CONFIG["database"]["analytics"]["analytics_url"])

## 1. Load the table `customer` table

Same connection pattern as the ingestion notebook - just reading this time, no writing.

In [2]:
df = pd.read_sql("SELECT * FROM bronze.pg_customers", engine)
print(df.shape)
df.head()


(99441, 7)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_source_file,_ingested_at
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,postgres/snapshots/table_name=customers/snapsh...,2026-09-03 07:26:59.329906
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,postgres/snapshots/table_name=customers/snapsh...,2026-09-03 07:26:59.329906
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,postgres/snapshots/table_name=customers/snapsh...,2026-09-03 07:26:59.329906
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,postgres/snapshots/table_name=customers/snapsh...,2026-09-03 07:26:59.329906
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,postgres/snapshots/table_name=customers/snapsh...,2026-09-03 07:26:59.329906


## 1. Shape and column types


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               99441 non-null  str           
 1   customer_unique_id        99441 non-null  str           
 2   customer_zip_code_prefix  99441 non-null  str           
 3   customer_city             99441 non-null  str           
 4   customer_state            99441 non-null  str           
 5   _source_file              99441 non-null  str           
 6   _ingested_at              99441 non-null  datetime64[us]
dtypes: datetime64[us](1), str(6)
memory usage: 21.3 MB


## 2. Missing values

Which columns have nulls, and how many? 

In [4]:
df.isnull().sum().sort_values(ascending=False)

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
_source_file                0
_ingested_at                0
dtype: int64

In [5]:
# Checking the length of customer id
set([len(id) for id in df['customer_id']])

{32}

## Seller Table



In [6]:
df = pd.read_sql("SELECT * FROM bronze.pg_sellers", engine)
print(df.info())
df.head()


<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   seller_id               3095 non-null   str           
 1   seller_zip_code_prefix  3095 non-null   str           
 2   seller_city             3095 non-null   str           
 3   seller_state            3095 non-null   str           
 4   _source_file            3095 non-null   str           
 5   _ingested_at            3095 non-null   datetime64[us]
dtypes: datetime64[us](1), str(5)
memory usage: 552.7 KB
None


,seller_id,seller_zip_code_prefix,seller_city,seller_state,_source_file,_ingested_at
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,postgres/snapshots/table_name=sellers/snapshot...,2026-09-03 07:26:59.858855
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,postgres/snapshots/table_name=sellers/snapshot...,2026-09-03 07:26:59.858855
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,postgres/snapshots/table_name=sellers/snapshot...,2026-09-03 07:26:59.858855
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,postgres/snapshots/table_name=sellers/snapshot...,2026-09-03 07:26:59.858855
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,postgres/snapshots/table_name=sellers/snapshot...,2026-09-03 07:26:59.858855


In [7]:
# Checking null values
df.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
_source_file              0
_ingested_at              0
dtype: int64

## Products table

In [8]:
df = pd.read_sql("SELECT * FROM bronze.pg_products", engine)
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   product_id                  32951 non-null  str           
 1   product_category_name       32341 non-null  str           
 2   product_name_lenght         32341 non-null  float64       
 3   product_description_lenght  32341 non-null  float64       
 4   product_photos_qty          32341 non-null  float64       
 5   product_weight_g            32949 non-null  float64       
 6   product_length_cm           32949 non-null  float64       
 7   product_height_cm           32949 non-null  float64       
 8   product_width_cm            32949 non-null  float64       
 9   _source_file                32951 non-null  str           
 10  _ingested_at                32951 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(7), str(3)
memory usage: 7.0 MB
Non

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_source_file,_ingested_at
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,postgres/snapshots/table_name=products/snapsho...,2026-09-03 07:26:59.740899
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,postgres/snapshots/table_name=products/snapsho...,2026-09-03 07:26:59.740899
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,postgres/snapshots/table_name=products/snapsho...,2026-09-03 07:26:59.740899
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,postgres/snapshots/table_name=products/snapsho...,2026-09-03 07:26:59.740899
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,postgres/snapshots/table_name=products/snapsho...,2026-09-03 07:26:59.740899


In [9]:
df.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
_source_file                    0
_ingested_at                    0
dtype: int64

In [10]:
df['product_category_name']

0                               perfumaria
1                                    artes
2                            esporte_lazer
3                                    bebes
4                    utilidades_domesticas
                       ...                
32946                     moveis_decoracao
32947    construcao_ferramentas_iluminacao
32948                      cama_mesa_banho
32949               informatica_acessorios
32950                      cama_mesa_banho
Name: product_category_name, Length: 32951, dtype: str

The source `pg_products` table stores `product_category_name` in **Portuguese**, which limits 
usability for English-speaking stakeholders and downstream reporting. To resolve this, we perform 
a **left join** against `api_translations`, a reference table mapping Portuguese category names to 
their English equivalents.

A **left join** (rather than an inner join) is used deliberately: we want to retain every product 
row from `pg_products`, even if no matching translation exists in `api_translations`. Using an inner 
join would silently drop products whose category has no translation entry — which would introduce 
data loss and undercount products in downstream analysis.


Since a translation may be missing, we apply a fallback chain using `COALESCE`:
1. Use the **English translation** (`product_category_name_english`) if available.
2. Otherwise, fall back to the **original Portuguese name** (`product_category_name`).
3. If both are missing, default to `'unknown'`.

This ensures `product_category` is **never null**, while still preferring the most usable value 
when it exists.|

In [11]:
df['product_category_name'].unique()

<ArrowStringArray>
[                                    'perfumaria',
                                          'artes',
                                  'esporte_lazer',
                                          'bebes',
                          'utilidades_domesticas',
                          'instrumentos_musicais',
                                     'cool_stuff',
                               'moveis_decoracao',
                               'eletrodomesticos',
                                     'brinquedos',
                                'cama_mesa_banho',
               'construcao_ferramentas_seguranca',
                         'informatica_acessorios',
                                   'beleza_saude',
                               'malas_acessorios',
                             'ferramentas_jardim',
                              'moveis_escritorio',
                                     'automotivo',
                                    'eletronicos',
            

In [12]:
# Check wheather the columns are actually float or just have .0 at the end

(df['product_length_cm'].dropna() % 1 != 0).any()

np.False_

## Order Table

In [13]:
df = pd.read_sql("SELECT * FROM bronze.pg_orders", engine)
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 19110 entries, 0 to 19109
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       19110 non-null  str           
 1   customer_id                    19110 non-null  str           
 2   order_status                   19110 non-null  str           
 3   order_purchase_timestamp       19110 non-null  datetime64[us]
 4   order_approved_at              19063 non-null  datetime64[us]
 5   order_delivered_carrier_date   18518 non-null  datetime64[us]
 6   order_delivered_customer_date  18219 non-null  datetime64[us]
 7   order_estimated_delivery_date  19110 non-null  datetime64[us]
 8   _source_file                   19110 non-null  str           
 9   _ingested_at                   19110 non-null  datetime64[us]
dtypes: datetime64[us](6), str(4)
memory usage: 4.2 MB
None


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_source_file,_ingested_at
0,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,2016-10-18 13:14:51,NaT,2016-10-20,postgres/batches/table_name=orders/batch_date=...,2026-09-03 07:26:11.983784
1,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,NaT,NaT,2016-10-28,postgres/batches/table_name=orders/batch_date=...,2026-09-03 07:26:11.983784
2,809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,NaT,NaT,2016-09-30,postgres/batches/table_name=orders/batch_date=...,2026-09-03 07:26:11.983784
3,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,postgres/batches/table_name=orders/batch_date=...,2026-09-03 07:26:11.983784
4,71303d7e93b399f5bcd537d124c0bcfa,b106b360fe2ef8849fbbd056f777b4d5,canceled,2016-10-02 22:07:52,2016-10-06 15:50:56,NaT,NaT,2016-10-25,postgres/batches/table_name=orders/batch_date=...,2026-09-03 07:26:11.983784


In [14]:
df.isnull().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 47
order_delivered_carrier_date     592
order_delivered_customer_date    891
order_estimated_delivery_date      0
_source_file                       0
_ingested_at                       0
dtype: int64

In [15]:
df.duplicated('order_id').sum()

np.int64(0)

##  Category values - like `order_status`

Check for inconsistent casing, stray whitespace, or unexpected values before deciding what the "clean" set of statuses should be.

In [16]:
df['order_status'].value_counts(dropna=False)

order_status
delivered      18215
shipped          299
unavailable      209
canceled         170
processing       123
invoiced          92
approved           2
Name: count, dtype: int64

**Rule this gives us:** trim and lowercase `order_status` so variants like `" Delivered"` and `"delivered"` collapse into one value.

## 5. Timestamp sanity checks

Min/max on each date column - looking for anything obviously wrong (a date in 1970, a delivery date before the purchase date, etc.).

In [17]:
ts_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in ts_cols:
    parsed = pd.to_datetime(df[col], errors='coerce')
    print(f"{col:35s} min={parsed.min()}  max={parsed.max()}  nulls={parsed.isna().sum()}")

order_purchase_timestamp            min=2016-09-04 21:15:19  max=2017-08-02 04:26:49  nulls=0
order_approved_at                   min=2016-09-15 12:16:38  max=2017-08-04 11:33:20  nulls=47
order_delivered_carrier_date        min=2016-10-08 10:34:01  max=2017-08-22 11:34:02  nulls=592
order_delivered_customer_date       min=2016-10-11 13:46:32  max=2017-12-04 18:36:29  nulls=891
order_estimated_delivery_date       min=2016-09-30 00:00:00  max=2017-10-19 00:00:00  nulls=0


## ORDER_ITEMS

In [18]:
df = pd.read_sql("SELECT * FROM bronze.pg_order_items", engine)
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 21363 entries, 0 to 21362
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             21363 non-null  str           
 1   order_item_id        21363 non-null  int64         
 2   product_id           21363 non-null  str           
 3   seller_id            21363 non-null  str           
 4   shipping_limit_date  21363 non-null  datetime64[us]
 5   price                21363 non-null  float64       
 6   freight_value        21363 non-null  float64       
 7   _source_file         21363 non-null  str           
 8   _ingested_at         21363 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(2), int64(1), str(4)
memory usage: 5.1 MB
None


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_source_file,_ingested_at
0,2e7a8482f6fb09756ca50c10d7bfc047,2,f293394c72c9b5fafd7023301fc21fc2,1554a68530182680ad5c8b042c3ab563,2016-10-26 18:25:19,32.90,31.67,postgres/batches/table_name=order_items/batch_...,2026-09-03 07:26:12.165178
1,2e7a8482f6fb09756ca50c10d7bfc047,1,c1488892604e4ba5cff5b4eb4d595400,1554a68530182680ad5c8b042c3ab563,2016-10-26 18:25:19,39.99,31.67,postgres/batches/table_name=order_items/batch_...,2026-09-03 07:26:12.165178
2,e5fa5a7210941f7d56d0208e4e071d35,1,f3c2d01a84c947b078e32bbef0718962,a425f92c199eb576938df686728acd20,2016-09-19 00:15:34,59.50,15.56,postgres/batches/table_name=order_items/batch_...,2026-09-03 07:26:12.165178
3,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,postgres/batches/table_name=order_items/batch_...,2026-09-03 07:26:12.165178
4,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,postgres/batches/table_name=order_items/batch_...,2026-09-03 07:26:12.165178


In [19]:
df.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
_source_file           0
_ingested_at           0
dtype: int64

In [20]:
df.duplicated(subset=['order_id','order_item_id']).sum()

np.int64(0)

## Order Payments

In [21]:
df = pd.read_sql("SELECT * FROM bronze.stream_order_payments", engine)
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 2494 entries, 0 to 2493
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   order_id              2494 non-null   str           
 1   payment_sequential    2494 non-null   int64         
 2   payment_type          2494 non-null   str           
 3   payment_installments  2494 non-null   int64         
 4   payment_value         2494 non-null   float64       
 5   _source_file          2494 non-null   str           
 6   _ingested_at          2494 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(2), str(3)
memory usage: 401.2 KB
None


,order_id,payment_sequential,payment_type,payment_installments,payment_value,_source_file,_ingested_at
0,dc28a299ebc4b33090b37f7419c9b99d,1,credit_card,2,111.21,external_s3/stream_event=order_payments/batch_...,2026-09-03 07:26:12.302552
1,487c1451b8fd7347d0e80e5aca887e91,2,voucher,1,30.00,external_s3/stream_event=order_payments/batch_...,2026-09-03 07:26:12.314846
2,7af7e9221757663bb19abb52d6acebf9,1,credit_card,1,485.72,external_s3/stream_event=order_payments/batch_...,2026-09-03 07:26:12.314846
3,a30cba7093267abeb0f8489d5c16c7af,1,credit_card,1,129.00,external_s3/stream_event=order_payments/batch_...,2026-09-03 07:26:12.314846
4,2c51c4240b040878ccc14a715bd52b0c,1,boleto,1,44.83,external_s3/stream_event=order_payments/batch_...,2026-09-03 07:26:12.314846


In [22]:
df.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
_source_file            0
_ingested_at            0
dtype: int64

In [23]:
df.duplicated(subset=['order_id','payment_sequential']).sum()

np.int64(0)

In [24]:
df['payment_type'].value_counts(dropna=False)

payment_type
credit_card    1778
boleto          533
voucher         164
debit_card       19
Name: count, dtype: int64

## Silver Order Payments

In [25]:
df = pd.read_sql("SELECT * FROM bronze.stream_order_reviews", engine)
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                1522 non-null   str           
 1   order_id                 1522 non-null   str           
 2   review_score             1522 non-null   int64         
 3   review_comment_title     0 non-null      object        
 4   review_comment_message   659 non-null    str           
 5   review_creation_date     1522 non-null   datetime64[us]
 6   review_answer_timestamp  1522 non-null   datetime64[us]
 7   _source_file             1522 non-null   str           
 8   _ingested_at             1522 non-null   datetime64[us]
dtypes: datetime64[us](3), int64(1), object(1), str(4)
memory usage: 343.6+ KB
None


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,_source_file,_ingested_at
0,ac6c5ae3b78be6ff4e00637f26c0c5fc,ffca67b9419980488c476f7f7f7bc070,2,None,NaN,2017-03-17,2017-03-21 01:39:16,external_s3/stream_event=order_reviews/batch_0...,2026-09-03 07:26:18.145260
1,7fba2de903adb964c3637f0cc76a86a3,d3704905336bdc5710b2bc0b55d9aad1,3,None,NaN,2017-03-18,2017-03-21 01:49:40,external_s3/stream_event=order_reviews/batch_0...,2026-09-03 07:26:18.145260
2,5a168c07ffcf24bb0ce4a01a2caa42cc,666a7787fe64880d1ad7dad0e4988ab3,1,None,Paguei por um produto que estava no anuncio e ...,2017-03-18,2017-03-21 01:49:47,external_s3/stream_event=order_reviews/batch_0...,2026-09-03 07:26:18.145260
3,c1b0c0be03c8ec647a965d68e4521360,69874f1392bc9898f11b59d0a735ce70,5,None,Boa loja eu indico ela ótimos produtos,2017-03-15,2017-03-21 01:50:31,external_s3/stream_event=order_reviews/batch_0...,2026-09-03 07:26:18.145260
4,8db42b5b590b91eb2b44c7a166cd00e4,b80d00e60a179d02f242e3f420023c78,5,None,NaN,2017-03-19,2017-03-21 03:26:43,external_s3/stream_event=order_reviews/batch_0...,2026-09-03 07:26:18.160129


In [26]:
df.isnull().sum()

review_id                     0
order_id                      0
review_score                  0
review_comment_title       1522
review_comment_message      863
review_creation_date          0
review_answer_timestamp       0
_source_file                  0
_ingested_at                  0
dtype: int64

In [27]:
df.duplicated(subset=['review_id','order_id','review_score','review_comment_title','review_comment_message']).sum()

np.int64(5)

In [28]:
df.duplicated().sum()

np.int64(0)

In [29]:
df['review_score'].unique()

array([2, 3, 1, 5, 4])

## Geo-location

In [30]:
df = pd.read_sql("SELECT * FROM bronze.api_geolocation", engine)
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 7 columns):
 #   Column                       Non-Null Count    Dtype         
---  ------                       --------------    -----         
 0   geolocation_zip_code_prefix  1000163 non-null  str           
 1   geolocation_lat              1000163 non-null  float64       
 2   geolocation_lng              1000163 non-null  float64       
 3   geolocation_city             1000163 non-null  str           
 4   geolocation_state            1000163 non-null  str           
 5   _source_file                 1000163 non-null  str           
 6   _ingested_at                 1000163 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(2), str(4)
memory usage: 122.4 MB
None


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,_source_file,_ingested_at
0,1037,-23.545621,-46.639292,sao paulo,SP,api/api_endpoint=geolocation/geolocation_lates...,2026-09-03 07:26:57.270148
1,1046,-23.546081,-46.644820,sao paulo,SP,api/api_endpoint=geolocation/geolocation_lates...,2026-09-03 07:26:57.270148
2,1046,-23.546129,-46.642951,sao paulo,SP,api/api_endpoint=geolocation/geolocation_lates...,2026-09-03 07:26:57.270148
3,1041,-23.544392,-46.639499,sao paulo,SP,api/api_endpoint=geolocation/geolocation_lates...,2026-09-03 07:26:57.270148
4,1035,-23.541578,-46.641607,sao paulo,SP,api/api_endpoint=geolocation/geolocation_lates...,2026-09-03 07:26:57.270148


In [31]:
df['geolocation_zip_code_prefix'].unique()

<ArrowStringArray>
[ '1037',  '1046',  '1041',  '1035',  '1012',  '1047',  '1013',  '1029',
  '1011',  '1032',
 ...
 '99965', '99990', '99930', '99960', '99925', '99955', '99970', '99910',
 '99920', '99952']
Length: 19015, dtype: str

In [32]:
df['geolocation_city'].unique()

<ArrowStringArray>
[            'sao paulo',             'são paulo', 'sao bernardo do campo',
               'jundiaí',       'taboão da serra',              'sãopaulo',
                    'sp',            'sa£o paulo',   'sao jose dos campos',
                'osasco',
 ...
       'ipiranga do sul',          'vila langaro',               'ciriaco',
      'floriano peixoto',              'erebango',                'ibiaçá',
  'santa cecilia do sul',               'ciríaco',               'estação',
          'vila lângaro']
Length: 8011, dtype: str

Customer and seller tables only store `zip_code_prefix` — never precise coordinates — so 
that's the finest grain we can ever join on. Keeping row-level lat/lng in the geolocation 
table adds no real precision and risks **join fan-out** (one prefix has many raw rows).

- **`AVG(lat)`, `AVG(lng)`** → a stable centroid per prefix, since all rows in a prefix are 
  already geographically close.
- **`MODE() city/state`** → the most common label per prefix, resolving spelling/casing 
  inconsistencies in the raw data.

Result: one clean row per `zip_code_prefix`, safe to join without duplication — matching 
the true grain of the source data.

In [33]:
def mode_or_nan(series):
    m = series.mode()
    return m.iloc[0] if not m.empty else None

grouped = df.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean'),
    city=('geolocation_city', mode_or_nan),
    state=('geolocation_state', mode_or_nan),
    count=('geolocation_zip_code_prefix', 'size')
).reset_index()

grouped

,geolocation_zip_code_prefix,lat,lng,city,state,count
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1002,-23.548146,-46.634979,sao paulo,SP,13
2,1003,-23.548994,-46.635731,sao paulo,SP,17
3,1004,-23.549799,-46.634757,sao paulo,SP,22
4,1005,-23.549456,-46.636733,sao paulo,SP,25
...,...,...,...,...,...,...
19010,99960,-27.953722,-52.025511,charrua,RS,5
19011,99965,-28.183372,-52.039850,agua santa,RS,6
19012,99970,-28.343766,-51.874689,ciriaco,RS,21
19013,99980,-28.389129,-51.843836,david canabarro,RS,26


In [34]:
grouped['city'].unique()

<ArrowStringArray>
[           'sao paulo',               'santos',              'caruara',
             'bertioga',          'sao vicente',              'guaruja',
              'cubatao',              'cubatão',        'sao sebastiao',
             'maresias',
 ...
      'ipiranga do sul',              'estacao',               'ibiaca',
 'santa cecilia do sul',         'vila langaro',              'charrua',
           'agua santa',              'ciriaco',      'david canabarro',
            'muliterno']
Length: 5829, dtype: str